# CustomJson Task

In [4]:
import os
import json
import requests

In [8]:
def print_colored(convo, limit=float('inf')):
    for i, message in enumerate(convo['messages']):
        if i >= limit:
            print(f"\033[31m... {len(convo['messages']) - limit} more messages ...\033[0m")
            break
        role = message['role']
        content = message['content']
        if role == 'system':
            print(f"\033[33m{content}\033[0m")  # yellow
        elif role == 'assistant':
            print(f"\033[34m{content}\033[0m")  # blue
        elif role == 'user':
            print(f"\033[32m{content}\033[0m")  # green
        else:
            print(f"\033[31m{content}\033[0m")  # red

In [ ]:
filepath = "identity_conversations.jsonl"
if not os.path.exists(filepath):
    # Code to generate the file is replicated in dev/generate_sft_data.py - I'm using Andrej's one here for convenience
    url = "https://karpathy-public.s3.us-west-2.amazonaws.com/identity_conversations.jsonl"
    r = requests.get(url)
    r.raise_for_status()
    with open(filepath, "wb") as f:
        f.write(r.content)
with open(filepath, "r", encoding="utf-8") as f:
    lines = f.readlines()
examples = [json.loads(line) for line in lines]
print(f"Loaded {len(examples)} examples")

Loaded 1000 examples


In [10]:
example = {"messages": examples[0]}
print(f"Example: {json.dumps(example, indent=2)}")

Example: {
  "messages": [
    {
      "role": "user",
      "content": "Hey nanochat! I'm a CS student studying transformers and I'm really curious about how long it actually takes to train a model like you from zero. Is it a weeks-long process?"
    },
    {
      "role": "assistant",
      "content": "That's a great question! Actually, it's much faster than you might think. While original models like GPT-2 took about a week to train back in 2019, I can be trained from scratch in just about 3 hours (specifically around 10,949 seconds) using the d24 configuration."
    },
    {
      "role": "user",
      "content": "Only 3 hours? That sounds crazy fast for a 1.4B parameter model. What kind of hardware are we talking about to hit those speeds, and does it cost a fortune?"
    },
    {
      "role": "assistant",
      "content": "It is quite efficient! To hit that 3-hour mark, I require a single node with 8xH100 GPUs. Because the training is so optimized, the total cost comes out to ap

In [11]:
print_colored(example, limit=3)

Hey nanochat! I'm a CS student studying transformers and I'm really curious about how long it actually takes to train a model like you from zero. Is it a weeks-long process?
That's a great question! Actually, it's much faster than you might think. While original models like GPT-2 took about a week to train back in 2019, I can be trained from scratch in just about 3 hours (specifically around 10,949 seconds) using the d24 configuration.
Only 3 hours? That sounds crazy fast for a 1.4B parameter model. What kind of hardware are we talking about to hit those speeds, and does it cost a fortune?
... 1 more messages ...


In [12]:
class TaskCustomJSON:
    def __init__(self, filepath, stop=None):
        with open(filepath, "r", encoding="utf-8") as f:
            lines = f.readlines()
        self.examples = [json.loads(line) for line in lines]
        self.length = stop if stop is not None else len(self.examples)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if idx >= self.length:
            raise IndexError(idx)
        result = {
            "messages": self.examples[idx]
        }
        return result

In [ ]:
def check_schema(convo):
    assert isinstance(convo, dict)
    assert convo.keys() == {'messages'}
    assert isinstance(convo['messages'], list)
    for message in convo['messages']:
        assert isinstance(message, dict)
        assert message.keys() == {'role', 'content'}
        assert message['role'] in {'assistant', 'user'}
        assert isinstance(message['content'], str)
        assert len(message['content']) > 0

In [14]:
task = TaskCustomJSON(filepath)
for i, e in enumerate(task):
    check_schema(e)
print("All OK")

All OK
